In [ ]:
# 0) Colab: mount Drive (train-data + runs) + clone code từ GitHub
# Chạy được bằng BẤT KỲ account nào (chính HOẶC phụ) — dùng CHUNG 1 path, không sửa code khi đổi account.
#
# YÊU CẦU 1 LẦN cho account phụ (account chính bỏ qua):
#   Mở link share (quyền Editor) -> chuột phải folder "recommender_train_colab"
#   -> Organize/Sắp xếp -> Add shortcut to Drive (Thêm lối tắt) -> My Drive.
#   Shortcut giữ nguyên tên => path /content/drive/MyDrive/recommender_train_colab
#   giống hệt account chính. Editor => ghi checkpoint/runs.csv thẳng vào folder gốc.
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

DRIVE_DIR = '/content/drive/MyDrive/recommender_train_colab'
assert os.path.isdir(DRIVE_DIR), (
    f"Khong thay {DRIVE_DIR}.\n"
    "Account phu: mo link share (Editor) trong Drive roi 'Add shortcut to My Drive' "
    "cho folder recommender_train_colab. Account chinh: kiem tra folder con o My Drive."
)

REPO = 'https://github.com/CrocodixD/anime-recommender.git'
CODE = '/content/recommender'

if os.path.exists(CODE):
    !cd {CODE} && git pull
else:
    !git clone {REPO} {CODE}

if not os.path.exists(f'{CODE}/train-data'):
    !cp -r "{DRIVE_DIR}/train-data" "{CODE}/train-data"

%cd {CODE}

In [ ]:
# 1) Imports + path (notebook chạy trong project_local/, model/ là package)
# imp shim: vài lib trên Colab (py3.12) còn import 'imp' đã bị gỡ khỏi stdlib
import sys, importlib
sys.modules['imp'] = importlib

import pathlib
HERE = pathlib.Path.cwd()
MODEL_DIR = HERE if HERE.name == 'model' else HERE / 'model'
sys.path.insert(0, str(MODEL_DIR))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import config, data, model as M, loss, metrics, train
print('torch', torch.__version__)

In [ ]:
# 2) Đường dẫn persistence trên Drive (checkpoint + history + leaderboard)
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/recommender_train_colab')
VERSION = 'v4'                                # đổi mỗi đợt experiment: v1, v2, v3...
RUNS_DIR = DRIVE / 'runs' / VERSION            # mỗi run: runs/<VERSION>/<run_name>/best.pt + history.json
RESULTS_CSV = DRIVE / 'runs.csv'               # leaderboard CHUNG mọi version (có cột 'version')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('version', VERSION, '· runs ->', RUNS_DIR)

In [ ]:
# 3) Base config T4 (Colab free: 2 core / 13GB RAM / 15GB VRAM). ckpt_dir set per-run ở helper (cell 5).
base_cfg = config.TwoTowerConfig(
    # model
    d=128,

    # loss
    tau=0.07,
    beta=1.0,

    # train
    lr=1e-3,
    batch_size=8192,
    epochs=1,
    hist_dropout=0.12,
    m_hardneg=5,

    # eval
    eval_every_steps=500,

    # infra
    num_workers=2,
    # subset=200_000,      # bỏ comment để chạy thử nhanh; None = full
)
print('device', base_cfg.device)   # 'cuda' trên T4
base_cfg

In [ ]:
# 4) Build module (sanity-check shape / param count)
spec, logq, item_table, users, mdl = train.build(base_cfg)
n_params = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
print(f'num_items={item_table.num_items}  num_users={users.num_users}  params={n_params:,}')
print('logq', logq.shape, 'finite', int(torch.isfinite(logq).sum()))

In [ ]:
# 5) Helpers thử nghiệm: mỗi run có tên riêng -> best.pt/history.json/row.json lưu trên Drive.
# Leaderboard (runs.csv) là artifact SUY RA từ mọi best.pt (rebuild_leaderboard) -> bền với
# run bị ngắt giữa chừng & nhiều account; không còn đọc-sửa-ghi đè 1 csv chung.
import json, time, dataclasses

def _cfg_from_ckpt(saved_cfg):
    """cfg trong ckpt mang path/device của session train CŨ (vd train_data=/content/project_local/...,
    device=cuda). Override về môi trường HIỆN TẠI (train-data đã copy về config.TRAIN_DATA, device máy
    này) -> build/eval lại không lỗi FileNotFoundError / cuda."""
    return dataclasses.replace(saved_cfg, device=config.auto_device(), train_data=config.TRAIN_DATA)

def load_run(run_name):
    """Load best.pt của 1 run -> (cfg, ckpt, model, users, logq) sẵn sàng eval/inference.
    Dựng model theo cfg LƯU TRONG checkpoint (gồm mọi override d/tau/...) -> khớp shape state_dict;
    chỉ thay path/device về môi trường hiện tại (xem _cfg_from_ckpt)."""
    ckpt = torch.load(RUNS_DIR / run_name / 'best.pt', weights_only=False)  # file kèm cfg -> cần False
    cfg = _cfg_from_ckpt(ckpt['cfg'])
    _, logq, _, users, model = train.build(cfg)
    model.load_state_dict(ckpt['model'])
    model.refresh_item_cache()
    return cfg, ckpt, model, users, logq

def eval_run(cfg, model, users, logq):
    """Eval full val + test cold-by-user -> dict phẳng để ghi 1 dòng leaderboard."""
    out = {}
    for split in ['val', 'test']:
        ds = data.ExamplesDataset(cfg.train_data, split)
        q = metrics.group_examples(ds.user_idx, ds.anime_idx)
        m = metrics.evaluate(model, users, q, logq, cfg.eval_ks)
        for k in cfg.eval_ks:
            out[f'{split}_recall@{k}'] = round(float(m[f'recall@{k}']), 4)
            out[f'{split}_ndcg@{k}']   = round(float(m[f'ndcg@{k}']), 4)
    return out

def _make_row(run_name, version, cfg, ckpt, model, users, logq, secs=None, ts=None):
    """1 dòng leaderboard từ (cfg, ckpt, model đã load) — eval lại val+test. Dùng chung run mới + khôi phục."""
    return {
        'run_name': run_name, 'version': version, 'time': ts,
        'd': cfg.d, 'tau': cfg.tau, 'beta': cfg.beta, 'lr': cfg.lr,
        'cosine_lr': cfg.cosine_lr, 'weight_decay': cfg.weight_decay,
        'batch_size': cfg.batch_size, 'epochs': cfg.epochs,
        'hist_dropout': cfg.hist_dropout, 'm_hardneg': cfg.m_hardneg,
        'use_item_id': cfg.use_item_id, 'id_dim': cfg.id_dim, 'id_dropout': cfg.id_dropout,
        'score_pool': cfg.score_pool,
        'history_pool': cfg.history_pool,
        'best_step': ckpt.get('step'), 'secs': secs,
        **eval_run(cfg, model, users, logq),
    }

def rebuild_leaderboard():
    """runs.csv = gom MỌI run có best.pt (mọi version). Nguồn 1 row, ưu tiên:
    row.json (cache durable) > dòng cũ trong runs.csv (migrate sang row.json) >
    eval lại từ best.pt (run bị ngắt giữa chừng -> dựng row, cache lại). KHÔNG train lại."""
    old = {}
    if RESULTS_CSV.exists():
        for r in pd.read_csv(RESULTS_CSV).to_dict('records'):
            old[(str(r.get('version')), str(r.get('run_name')))] = r
    rows = []
    for ckpt_path in sorted((DRIVE / 'runs').glob('*/*/best.pt')):
        run_dir = ckpt_path.parent
        run_name, version = run_dir.name, run_dir.parent.name
        row_file = run_dir / 'row.json'
        if row_file.exists():
            row = json.loads(row_file.read_text())
        elif (version, run_name) in old:
            row = old[(version, run_name)]
            row_file.write_text(json.dumps(row, default=float))         # migrate dòng cũ -> durable
        else:                                                           # run bị ngắt -> eval lại từ best.pt
            ck = torch.load(ckpt_path, weights_only=False)
            cfg = _cfg_from_ckpt(ck['cfg'])
            _, logq, _, users, model = train.build(cfg)
            model.load_state_dict(ck['model']); model.refresh_item_cache()
            row = _make_row(run_name, version, cfg, ck, model, users, logq)
            row_file.write_text(json.dumps(row, default=float))
            print(f"  recovered {version}/{run_name} (eval từ best.pt)")
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)
    return df.sort_values('test_recall@100', ascending=False)

def run_experiment(run_name, **overrides):
    """Clone base_cfg + override hyperparam -> train -> eval -> ghi row.json + rebuild runs.csv.
    overrides là field của TwoTowerConfig (vd use_item_id=True, id_dim=128, tau=0.05, epochs=2)."""
    cfg = dataclasses.replace(base_cfg, **overrides)
    cfg.ckpt_dir = RUNS_DIR / run_name
    t0 = time.time()
    model, history = train.fit(cfg)
    secs = time.time() - t0
    # history -> json (plot lại sau / sống sót disconnect)
    (cfg.ckpt_dir / 'history.json').write_text(json.dumps(history, default=float))
    # eval lại từ best.pt (model tốt nhất, không phải step cuối) -> row.json (durable per-run)
    cfg, ckpt, model_e, users_e, logq_e = load_run(run_name)
    row = _make_row(run_name, VERSION, cfg, ckpt, model_e, users_e, logq_e,
                    secs=round(secs), ts=pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'))
    (cfg.ckpt_dir / 'row.json').write_text(json.dumps(row, default=float))
    print(f"\n✓ {run_name}: test recall@100={row['test_recall@100']}")
    rebuild_leaderboard()                              # runs.csv = gom mọi run (durable, tự lành)
    return cfg, history, row

In [ ]:
# 6) Chạy 1 thử nghiệm. Đổi tên + kwargs cho mỗi lần (checkpoint KHÔNG ghi đè nhau).
# Attention pooling: history_pool='attn' (vs 'mean' mặc định). So decisive vs itemid128_ep6_d01 (mean)=0.394.
cfg, history, row = run_experiment('itemid128_ep6_attn', use_item_id=True, id_dim=128, id_dropout=0.1, epochs=6, history_pool='attn')

# Quick-check 1 epoch (~11ph) — attn vs control mean cùng config (control d01-ep1 trước giờ còn thiếu):
# cfg, history, row = run_experiment('itemid128_attn', use_item_id=True, id_dim=128, id_dropout=0.1, history_pool='attn')
# cfg, history, row = run_experiment('itemid128_d01',  use_item_id=True, id_dim=128, id_dropout=0.1, history_pool='mean')

In [ ]:
# 7) Đường cong training loss (raw + moving-average) — dùng `history` trả từ run_experiment
ls = np.array(history['loss_steps']); lv = np.array(history['loss_vals'])
plt.figure(figsize=(8, 4))
plt.plot(ls, lv, alpha=0.3, label='loss')
w = max(1, len(lv) // 50)
if len(lv) >= w and w > 1:
    ma = np.convolve(lv, np.ones(w) / w, mode='valid')
    plt.plot(ls[w - 1:], ma, label=f'MA({w})')
plt.xlabel('step'); plt.ylabel('InfoNCE loss'); plt.title('Training loss')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# 8) Đường cong val metric theo step (recall@K, ndcg@K). Retriever -> để ý recall@100.
es = history['eval_steps']; em = history['eval_metrics']
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k in cfg.eval_ks:
    ax[0].plot(es, [m[f'recall@{k}'] for m in em], marker='o', label=f'recall@{k}')
    ax[1].plot(es, [m[f'ndcg@{k}'] for m in em], marker='o', label=f'ndcg@{k}')
ax[0].set_title('Val recall@K'); ax[0].set_xlabel('step')
ax[1].set_title('Val ndcg@K'); ax[1].set_xlabel('step')
for a in ax: a.legend(); a.grid(alpha=0.3)
plt.show()

In [ ]:
# 9) Leaderboard: gom MỌI run có best.pt (kể cả run bị ngắt giữa chừng -> eval lại từ best.pt).
# Headline retriever = test_recall@100. Lần đầu có thể eval lại vài run thiếu row.json (chậm chút), sau đó cache.
rebuild_leaderboard()